# Round 2: Logistic Regression (Primary & Secondary)

**Goal**: Train Logistic Regression models for both Primary and Secondary emotion classification.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

sns.set(style='whitegrid')
reports_dir = 'round2/reports'
os.makedirs(reports_dir, exist_ok=True)

## 1. Load Data

In [ ]:
train_df = pd.read_excel('round2/train.xlsx')
val_df = pd.read_excel('round2/val.xlsx')

X_train = train_df['cleaned_poem'].astype(str)
X_val = val_df['cleaned_poem'].astype(str)

# Primary Labels
y_train_p = train_df['primary_id']
y_val_p = val_df['primary_id']

# Secondary Labels
y_train_s = train_df['secondary_id']
y_val_s = val_df['secondary_id']

# Load Maps
with open('round2/label_maps.json', 'r') as f:
    maps = json.load(f)
p_map = {v: k for k, v in maps['primary_map'].items()}
s_map = {v: k for k, v in maps['secondary_map'].items()}

## 2. Feature Extraction

In [ ]:
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=5, sublinear_tf=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
print(f"Feature Shape: {X_train_vec.shape}")

## 3. Train & Evaluate Helper Function

In [ ]:
def train_evaluate(y_train, y_val, label_map, prefix):
    print(f"\nTraining Logistic Regression for {prefix}...")
    model = LogisticRegression(class_weight='balanced', multi_class='multinomial', max_iter=2000, n_jobs=-1)
    model.fit(X_train_vec, y_train)
    
    y_pred = model.predict(X_val_vec)
    
    acc = accuracy_score(y_val, y_pred)
    macro = f1_score(y_val, y_pred, average='macro')
    weighted = f1_score(y_val, y_pred, average='weighted')
    
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {macro:.4f}")
    
    # Save Metrics
    metrics = {'accuracy': acc, 'macro_f1': macro, 'weighted_f1': weighted}
    with open(f'{reports_dir}/{prefix}_metrics.json', 'w') as f:
        json.dump(metrics, f, indent=4)
        
    # Confusion Matrix
    cm = confusion_matrix(y_val, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, cmap='Greens', fmt='d')
    plt.title(f'{prefix} LR Confusion Matrix')
    plt.savefig(f'{reports_dir}/{prefix}_confusion_matrix.png')
    plt.show()
    
    # Save Errors
    res = pd.DataFrame({'True': y_val, 'Pred': y_pred})
    res['True_Label'] = res['True'].map(label_map)
    res['Pred_Label'] = res['Pred'].map(label_map)
    errors = res[res['True'] != res['Pred']]
    errors.to_csv(f'{reports_dir}/{prefix}_errors.csv', index=False)

## 4. Run Task 1: Primary Classification

In [ ]:
train_evaluate(y_train_p, y_val_p, p_map, 'primary_lr')

## 5. Run Task 2: Secondary Classification

In [ ]:
train_evaluate(y_train_s, y_val_s, s_map, 'secondary_lr')